In [3]:
# ReLU関数
import numpy as np

def relu(x):
    return np.maximum(x, 0)

def deriv_relu(x):
    return (x > 0).astype(x.dty)

In [4]:
# Sigmoid関数
def sigmoid(x):
    return np.exp(np.minimum(x, 0)) / (1 + np.exp(- np.abs(x)))

def deriv_sigmoid(x):
    return sigmoid(x) * (1 - sigmoid(x))

In [5]:
def np_log(x):
    return np.log(np.clip(x, 1e-10, 1e+10))

In [6]:
def train_xor(x, t, eps):
    """
    :param x: np.ndarray, input data, (batch_size, input_dim)
    :param t: np.ndarray, ground-truth labels, (batch_size, output_dim)
    :param eps: float, learning rate
    """
    global W1, b1, W2, b2

    batch_size = x.shape[0]

    # Forward propagation
    # np.matmul()：入力xと重みW1の行列積を計算
    # バイアスb1を足す
    u1 = np.matmul(x, W1) + b1  # (batch_size, hidden_dim)
    # 前の層で求めた値をReLU関数に通して、隠れ層の出力h1を求める
    h1 = relu(u1)

    # 隠れ層h1を入力として、重みW2とバイアスb2を使ったAffine変換を行う
    u2 = np.matmul(h1, W2) + b2  # (batch_size, output_dim)
    # シグモイド関数に通して、出力を確率に変換
    y = sigmoid(u2)


    # Compute loss
    # 2クラス交差エントロピーの計算
    cost = (- t * np_log(y) - (1 - t) * np_log(1 - y)).mean()

    # Backpropagation
    # シグモイド関数の逆伝播
    # 予測値から正解データを引く
    delta_2 = y - t  
    # ReLU関数の逆伝播
    # deriv_relu(u1)：ReLUの微分
    # W2.T：重みW2の転置行列
    delta_1 = deriv_relu(u1) * np.matmul(delta_2, W2.T)  

    # Compute gradients
    # 1層目の重みとバイアスの逆伝播
    dW1 = np.matmul(x.T, delta_1) / batch_size  
    db1 = np.matmul(np.ones(batch_size), delta_1) / batch_size   

    # 2層目の重みとバイアスの逆伝播
    dW2 = np.matmul(h1.T, delta_2) / batch_size  
    db2 = np.matmul(np.ones(batch_size), delta_2) / batch_size   

    # Update parameters
    # パラメータの更新
    W1 -= eps * dW1 
    b1 -= eps * db1 

    W2 -= eps * dW2 
    b2 -= eps * db2 

    return cost

def valid_xor(x, t):
    global W1, b1, W2, b2

    # Forward propagation
    u1 = np.matmul(x, W1) + b1
    h1 = relu(u1)

    u2 = np.matmul(h1, W2) + b2
    y = sigmoid(u2)

    # Compute loss
    cost = (- t * np_log(y) - (1 - t) * np_log(1 - y)).mean() 

    return cost, y

In [7]:
def softmax(x):
    x -= x.max(axis=1, keepdims=True)  # Avoid overflow
    x_exp = np.exp(x)
    return x_exp / np.sum(x_exp, axis=1, keepdims=True)


def deriv_softmax(x):
    return softmax(x) * (1 - softmax(x))

In [ ]:
class Dense:
    # in_dim：入力の次元数
    # out_dim：出力の次元数
    # function：活性化関数
    # deriv_function：活性化関数の微分
    def __init__(self, in_dim, out_dim, function, deriv_function):
        # np.random.uniform：一様分布から乱数を生成するメソッド
        self.W = np.random.uniform(low=-0.08, high=0.08,
                                   size=(in_dim, out_dim)).astype("float64")
        # バイアスの初期化、0で埋める
        self.b = np.zeros(out_dim).astype("float64")
        # この層で使う活性化関数とその微分をインスタンス変数として保持
        self.function = function
        self.deriv_function = deriv_function
        
        # x：入力データを保持
        self.x = None
        # u：活性化関数を通す前の計算結果を保持
        self.u = None

        # 逆伝播で計算される重みとバイアスの勾配を保持
        self.dW = None
        self.db = None

        # ([self.W.size, self.b.size])：重みの要素数とバイアスの要素数を並べたリストを作成
        # np.cumsum()：累積和を計算する関数。リストの値を順番に足し合わせる
        self.params_idxs = np.cumsum([self.W.size, self.b.size])

    def __call__(self, x):
        """
        Method that performs forward propagation.
        x: (batch_size, in_dim_{j})
        h: (batch_size, out_dim_{j})
        """
        self.x = x
        self.u = np.matmul(self.x, self.W) + self.b
        h = self.function(self.u)
        return h

    def b_prop(self, delta, W):
        """
        Method that performs backpropagation.
        delta (=delta_{j+1}): (batch_size, out_dim_{j+1})
        W (=W_{j+1}): (out_dim_{j}, out_dim_{j+1})
        self.delta (=delta_{j}): (batch_size, out_dim_{j})
        """
        self.delta = self.deriv_function(self.u) * np.matmul(delta, W.T) # WRITE ME
        return self.delta

    def compute_grad(self):
        """
        Method that computes gradients.
        self.x: (batch_size, in_dim_{j})
        self.delta: (batch_size, out_dim_{j})
        self.dW: (in_dim_{j}, out_dim_{j})
        self.db: (out_dim_{j})
        """
        batch_size = self.delta.shape[0]

        self.dW = np.matmul(self.x.T, self.delta) / batch_size # WRITE ME
        self.db = np.matmul(np.ones(batch_size), self.delta) / batch_size # WRITE ME

    def get_params(self):
        return np.concatenate([self.W.ravel(), self.b], axis=0)

    def set_params(self, params):
        """
        params: List[np.ndarray, np.ndarray]
            The first element is the weight matrix W: (in_dim, out_dim), and the second element is the bias vector: (out_dim,)
        """
        _W, _b = np.split(params, self.params_idxs)[:-1]
        self.W = _W.reshape(self.W.shape)
        self.b = _b

    def get_grads(self):
        return np.concatenate([self.dW.ravel(), self.db], axis=0)